In [1]:
import json
from pathlib import Path

pipeline_notebook_path = Path.cwd() / "rag_pipeline.ipynb"
if not pipeline_notebook_path.is_file():
    raise FileNotFoundError(f"Pipeline notebook not found: {pipeline_notebook_path}")

pipeline_notebook = json.loads(pipeline_notebook_path.read_text(encoding="utf-8"))
pipeline_code_cells = [
    cell for cell in pipeline_notebook["cells"] if cell["cell_type"] == "code"
]
if not pipeline_code_cells:
    raise ValueError(f"No code cells found in {pipeline_notebook_path}")

pipeline_namespace = {"__name__": "rag_pipeline_notebook"}
pipeline_source = "".join(pipeline_code_cells[0]["source"])
exec(compile(pipeline_source, str(pipeline_notebook_path), "exec"), pipeline_namespace)

retriever_func = pipeline_namespace["retriever_func"]
generate_answers = pipeline_namespace["generate_answers"]

d:\rag_project\basic_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 882.99it/s]


In [2]:
from pathlib import Path

db_root = Path("faiss_db")
db_candidates = [
    path
    for path in [db_root, *db_root.glob("*/")]
    if (path / "index.faiss").is_file() and (path / "index.pkl").is_file()
]
if not db_candidates:
    raise FileNotFoundError(f"No existing FAISS index found under {db_root.resolve()}")

db_path = max(
    db_candidates,
    key=lambda path: (path / "index.faiss").stat().st_mtime,
)
print(f"Using FAISS index: {db_path}")

evaluation_data = [
    {
        "question": "What is the full form of RAG?",
        "expected_keyword": "retrieval augmented generation",
    },
    {
        "question": "What is one major disadvantage of using only an LLM?",
        "expected_keyword": "hallucinat",
    },
    {
        "question": "Why do RAG systems use a vector database?",
        "expected_keyword": "vector",
    },
]

def run_evaluation(test_data, k=4):
    if not test_data:
        raise ValueError("Evaluation requires at least one test question.")
    if k < 1:
        raise ValueError("k must be at least 1.")
    if k > 4:
        raise ValueError("The pipeline retriever returns at most four documents.")

    results = []
    for index, item in enumerate(test_data, start=1):
        question = item["question"]
        expected_keyword = item["expected_keyword"].lower()
        retrieved_docs = retriever_func(question, persist_directory=str(db_path))[:k]
        retrieved_context = " ".join(doc.page_content.lower() for doc in retrieved_docs)
        retrieval_hit = expected_keyword in retrieved_context
        answer = generate_answers(question, persist_directory=str(db_path))

        results.append(
            {
                "question": question,
                "retrieval_hit": retrieval_hit,
                "answer": answer,
            }
        )
        status = "PASS" if retrieval_hit else "FAIL"
        print(f"\n[Test {index}/{len(test_data)}] Retrieval {status}: {question}")
        print(f"Expected keyword: {expected_keyword}")
        print(f"Generated answer:\n{answer.strip()}")

    hits = sum(result["retrieval_hit"] for result in results)
    hit_rate = hits / len(results) * 100
    print(f"\nRetrieval hit rate (top-{k}): {hits}/{len(results)} ({hit_rate:.2f}%)")
    return {"hit_rate_percent": hit_rate, "results": results}

evaluation_results = run_evaluation(evaluation_data)

Using FAISS index: faiss_db\e9c432cfe7694e1285f2c114b0b8cddf
Asking the LLM: What is the full form of RAG?

$$$ FINAL ANSWER $$$

[Test 1/3] Retrieval PASS: What is the full form of RAG?
Expected keyword: retrieval augmented generation
Generated answer:
*   The full form of RAG is Retrieval Augmented Generation.
    *   Source: https://www.youtube.com/watch?v=fZM3oX4xEyg
    *   Source: https://www.youtube.com/watch?v=Chl-cRcwVpA
Asking the LLM: What is one major disadvantage of using only an LLM?

$$$ FINAL ANSWER $$$

[Test 2/3] Retrieval PASS: What is one major disadvantage of using only an LLM?
Expected keyword: hallucinat
Generated answer:
One major disadvantage of using only an LLM is:

*   **Hallucination**: The LLM may generate its own answers, even if it lacks the actual knowledge, to avoid appearing uninformed. This means it can produce convincing but incorrect information.
    *   Source: https://www.youtube.com/watch?v=fZM3oX4xEyg
Asking the LLM: Why do RAG systems use a ve

In [3]:
from rag_pipeline import retriever_func, generate_answers

# 1. Point directly to your existing FAISS database folder
db_path = "faiss_db/e9c432cfe7694e1285f2c114b0b8cddf"

# 2. Simple dictionary of {Question: Expected Keyword}
question_dict = {
    "What is the full form of RAG?": "retrieval augmented generation",
    "What is one major disadvantage of using only an LLM?": "hallucinat",
    "Why do RAG systems use a vector database?": "vector",
}

# 3. Simple loop to check evaluation
hits = 0

for question, expected_keyword in question_dict.items():
    # Check Retriever
    docs = retriever_func(question, persist_directory=db_path)
    context = " ".join(doc.page_content.lower() for doc in docs)
    
    if expected_keyword.lower() in context:
        hits += 1
        print(f"[PASS] Question: {question}")
    else:
        print(f"[FAIL] Question: {question}")
        
    # Check Generator Answer
    answer = generate_answers(question, persist_directory=db_path)
    print(f"Answer:\n{answer}\n" + "-" * 50)

# Final Score Recall@k
hit_rate = (hits / len(question_dict)) * 100
print(f"\nFinal Evaluation Score (Hit Rate): {hits}/{len(question_dict)} ({hit_rate:.2f}%)")

[PASS] Question: What is the full form of RAG?
Asking the LLM: What is the full form of RAG?

$$$ FINAL ANSWER $$$
Answer:
The full form of RAG is:

*   **Retrieval Augmented Generation** (Source: https://www.youtube.com/watch?v=fZM3oX4xEyg, https://www.youtube.com/watch?v=Chl-cRcwVpA)
--------------------------------------------------
[PASS] Question: What is one major disadvantage of using only an LLM?
Asking the LLM: What is one major disadvantage of using only an LLM?

$$$ FINAL ANSWER $$$
Answer:
One major disadvantage of using only an LLM is:

*   **Hallucination:** LLMs may generate answers even when they lack the actual knowledge, creating content that appears believable but is not factual. This happens because the LLM tries to generate an answer rather than admitting it doesn't know.
    *   Source: https://www.youtube.com/watch?v=fZM3oX4xEyg
--------------------------------------------------
[PASS] Question: Why do RAG systems use a vector database?
Asking the LLM: Why do RAG

In [11]:
import os
import json
import numpy as np

# 0. Auto-sync rag_pipeline.ipynb -> rag_pipeline.py so imports always work
if not os.path.exists("rag_pipeline.py") and os.path.exists("rag_pipeline.ipynb"):
    with open("rag_pipeline.ipynb", "r", encoding="utf-8") as f:
        nb = json.load(f)
    code_cells = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
    with open("rag_pipeline.py", "w", encoding="utf-8") as f:
        f.write("\n\n".join(code_cells))

from rag_pipeline import (
    embedding_model,
    get_vector_store,
    retriever_func,
    generate_answers
)

# Point to the exact FAISS database created in your rag_pipeline output
db_path = "./faiss_db/e9c432cfe7694e1285f2c114b0b8cddf"


# ==========================================================
# 1. EMBEDDINGS EVALUATION (all-MiniLM-L6-v2)
# ==========================================================
print("=" * 60)
print("1. EMBEDDINGS EVALUATION")
print("=" * 60)

v_query = np.array(embedding_model.embed_query("What is RAG?"))
v_similar = np.array(embedding_model.embed_query("Retrieval Augmented Generation"))
v_unrelated = np.array(embedding_model.embed_query("How to cook pasta"))

cosine_sim = lambda a, b: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Embedding Dimensions         : {len(v_query)}")
print(f"Semantic Similarity (Related): {cosine_sim(v_query, v_similar):.4f}")
print(f"Semantic Similarity (Unrel.) : {cosine_sim(v_query, v_unrelated):.4f}\n")


# ==========================================================
# 2. VECTOR DATABASE (FAISS) EVALUATION
# ==========================================================
print("=" * 60)
print("2. VECTOR DATABASE (FAISS) EVALUATION")
print("=" * 60)

vector_store = get_vector_store(db_path)
index_size = vector_store.index.ntotal

print(f"Database Path                : {db_path}")
print(f"Successful Loading           : PASS")
print(f"Index Size (Total Chunks)    : {index_size} (Expected: 300)")
print(f"Document Availability        : {'PASS' if index_size > 0 else 'FAIL'}\n")


# ==========================================================
# 3 & 4. RETRIEVER & GENERATED OUTPUT EVALUATION
# ==========================================================
print("=" * 60)
print("3 & 4. RETRIEVER & GENERATED OUTPUT EVALUATION")
print("=" * 60)

# Test dataset matched to your 8 loaded Krish Naik RAG videos
test_data = [
    {
        "question": "What is the full form of RAG?",
        "expected_keyword": "retrieval augmented generation",
        "expected_source": "https://www.youtube.com/watch?v=fZM3oX4xEyg"
    },
    {
        "question": "What are the disadvantages of just using the LLM model?",
        "expected_keyword": "hallucinat",
        "expected_source": "https://www.youtube.com/watch?v=fZM3oX4xEyg"
    },
    {
        "question": "Why do we use a vector database in a RAG pipeline?",
        "expected_keyword": "vector",
        "expected_source": "https://www.youtube.com/watch?v=MykcjWPJ6T4"
    }
]

k = 4  # Matches search_kwargs={'k': 4} in your retriever_func
hits = 0
precision_list = []
recall_list = []
mrr_list = []
faithful_count = 0
relevant_count = 0
correct_count = 0

for i, item in enumerate(test_data, 1):
    q = item["question"]
    kw = item["expected_keyword"].lower()
    
    # --- RETRIEVER CHECK (`retriever_func`) ---
    retrieved_docs = retriever_func(q, persist_directory=db_path)[:k]
    
    # Check relevance of each of the 4 retrieved chunks
    rel_flags = [kw in doc.page_content.lower() for doc in retrieved_docs]
    
    # Hit Rate@K & Recall@K
    hit = int(any(rel_flags))
    hits += hit
    recall_list.append(hit)
    
    # Precision@K (Relevant chunks retrieved / k)
    precision_k = sum(rel_flags) / k
    precision_list.append(precision_k)
    
    # MRR (1 / rank of first relevant chunk)
    rr = next((1.0 / rank for rank, is_rel in enumerate(rel_flags, 1) if is_rel), 0.0)
    mrr_list.append(rr)
    
    # --- GENERATED OUTPUT CHECK (`generate_answers`) ---
    answer = generate_answers(q, persist_directory=db_path)
    ans_lower = answer.lower()
    
    # Faithfulness: Did NOT use the external outsourcing disclaimer from your PromptTemplate
    is_faithful = "note: this part is outsourcing from external knowledge" not in ans_lower
    if is_faithful:
        faithful_count += 1
        
    # Answer Relevance: Addressed the query and cited a YouTube video source
    is_relevant = "youtube.com/watch" in ans_lower and len(answer.strip()) > 20
    if is_relevant:
        relevant_count += 1
        
    # Answer Correctness: Contains the expected core concept
    is_correct = kw in ans_lower
    if is_correct:
        correct_count += 1

    print(f"\n[Test {i}] Question: {q}")
    print(f"  Retriever -> Hit@{k}: {bool(hit)} | Precision@{k}: {precision_k:.2f} | Recall@{k}: {hit:.2f} | RR: {rr:.2f}")
    print(f"  Generator -> Faithfulness: {'PASS' if is_faithful else 'FAIL'} | Relevance: {'PASS' if is_relevant else 'FAIL'} | Correctness: {'PASS' if is_correct else 'FAIL'}")
    print(f"  Answer    :\n{answer.strip()}")
    print("-" * 60)


# ==========================================================
# 5. OVERALL RAG SUMMARY
# ==========================================================
n = len(test_data)
print("\n" + "=" * 60)
print("5. OVERALL RAG EVALUATION SUMMARY")
print("=" * 60)
print(f"Embeddings        -> Dimensions: {len(v_query)} | Semantic Separation: PASS")
print(f"Vector DB (FAISS) -> Loaded: PASS | Total Chunks: {index_size}")
print(f"Retriever (k={k})   -> Hit Rate@{k}: {(hits/n)*100:.1f}% | Recall@{k}: {np.mean(recall_list)*100:.1f}% | Precision@{k}: {np.mean(precision_list)*100:.1f}% | MRR: {np.mean(mrr_list):.2f}")
print(f"Generated Output  -> Faithfulness: {(faithful_count/n)*100:.1f}% | Answer Relevance: {(relevant_count/n)*100:.1f}% | Correctness: {(correct_count/n)*100:.1f}%")
print("=" * 60)

ImportError: cannot import name 'embedding_model' from 'rag_pipeline' (d:\rag_project\basic_rag\rag_pipeline.py)